In [ ]:
import gzip

path = "../data/raw/simplewiki/latest/page.sql.gz"

total_lines = 0
insert_lines = 0
first_insert = None

with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        total_lines += 1
        if line.startswith("INSERT INTO"):
            insert_lines += 1
            if first_insert is None:
                first_insert = line
print("total lines: ", total_lines)
print("INSERT lines: ", insert_lines)
print("length of first insert line: ", len(first_insert))
print(first_insert[:300])

In [ ]:
with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        if line.startswith("INSERT INTO"):
            break

    print("---first 3 rows after insert---")
    for _ in range(3):
        print(repr(next(f)))

    for line in f:
        if line.rstrip("\n").endswith(";"):
            print("---last row of the block---")
            print(repr(line))
            print("---line after it---")
            print(repr(next(f)))
            break

In [ ]:
from collections import Counter


def naive_split(line):
    body = line.rstrip("\n")
    body = body[1:-2]
    return body.split(",")


value_counts = Counter()
broken_examples = []

with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        if not line.startswith("("):
            continue
        values = naive_split(line)
        value_counts[len(values)] += 1
        if len(values) != 12 and len(broken_examples) < 5:
            broken_examples.append(line)

print("rows found: ", sum(value_counts.values()))
print("values per row: ", value_counts)
for line in broken_examples:
    print(repr(line))

In [ ]:
import csv


def split_row(line):
    body = line.rstrip("\n")[1:-2]
    reader = csv.reader([body], quotechar="'", escapechar="\\")
    return next(reader)


value_counts = Counter()
broken_examples = []

with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        if not line.startswith("("):
            continue
        values = split_row(line)
        value_counts[len(values)] += 1
        if len(values) != 12 and len(broken_examples) < 5:
            broken_examples.append(line)

print("rows found:", sum(value_counts.values()))
print("values per row:", value_counts)
for line in broken_examples:
    print(repr(line))

print(split_row("(2135,0,'Athens,_Greece',1,0,0.6,'2026','2024',1,56,'wikitext',NULL),\n"))

In [ ]:
def parse_page_row(line):
    values = split_row(line)
    page_id = int(values[0])
    namespace = int(values[1])
    title = values[2]
    is_redirect = int(values[3]) == 1
    return page_id, namespace, title, is_redirect


print(parse_page_row("(2135,0,'Athens,_Greece',1,0,0.6,'2026','2024',1,56,'wikitext',NULL),\n"))

namespace_counts = Counter()
articles = 0
redirects = 0

with gzip.open(path, "rt", encoding="utf-8") as f:
    for line in f:
        if not line.startswith("("):
            continue
        page_id, namespace, title, is_redirect = parse_page_row(line)
        namespace_counts[namespace] += 1
        if namespace == 0:
            if is_redirect:
                redirects += 1
            else:
                articles += 1

print("articles (ns 0, no redirect):", articles)
print("redirects (ns 0):            ", redirects)
print("pages per namespace:", sorted(namespace_counts.items()))

<span style="background-color: white; color: black;">
The graph will have 286,136 nodes.
There are 115,576 redirects. So about 29% of namespace 0 are redirects.
</span>


In [ ]:
%load_ext autoreload
%autoreload 2
from wikigraph.ingest.sql_dump import iter_pages

In [ ]:
articles = 0
redirects = 0
for _page_id, namespace, _title, is_redirect in iter_pages(path):
    if namespace == 0:
        if is_redirect:
            redirects += 1
        else:
            articles += 1
print(articles, redirects)

<span style="background-color: white; color: black;">
Ingest part is working.
</span>